# Lab 7 — Support Vector Machines
**Coverage:** Chapter 14

This notebook is one of the ten course labs. Complete the core activities in order; transfer activities are optional extensions inside the same lab and do not create additional lab numbers.


## Part A — Standardized SVM tuning on breast-cancer data
**Core activity.**


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "breast_cancer_dataset" / "data.csv"
df = pd.read_csv(train_path)

In [ ]:
X = df.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y = (df["diagnosis"] == "M").astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC()),
])
search = GridSearchCV(
    pipe,
    param_grid={
        "model__kernel": ["linear", "rbf"],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.01, 0.1],
    },
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc",
    n_jobs=-1,
)

In [ ]:
search.fit(X_train, y_train)
score = search.decision_function(X_test)
pred = search.predict(X_test)
print("best:", search.best_params_)
print("accuracy:", round(accuracy_score(y_test, pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, score), 3))